# ABI Baseline vs Cost-Weighted GEC — Converter vs MCI Comparison

Adapts the Abnormality Index (ABI) from Stoecklein et al. (2020) to Schaefer 200-ROI
parcellated correlation matrices.
Uses a strict binary setup from `__v4__/metadata`: converters = `converter`, non-converters = `mci`.
Compares ABI-based classification against the Cost-Weighted GEC model on the same strict splits.

In [1]:
import json
import numpy as np
import os
import pandas as pd
import seaborn as sns
import warnings

from pathlib import Path
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from datetime import datetime

sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=FutureWarning)

## Configuration

In [2]:
WB_ROOT = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v3__/matrices')
METADATA_DIR = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/metadata')
SPLITS_DIR = METADATA_DIR / 'splits_gec'
COHORTS_CSV = METADATA_DIR / 'cohorts.csv'

GEC_CHECKPOINT_DIR = Path('/mnt/e/fyassine/ad-early-detection/MODEL/notebooks/checkpoints_cost_weighted_gec_whole_brain')

OUTPUT_DIR = Path('/mnt/e/fyassine/ad-early-detection/DCI/notebooks/artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILE_VARIANT = 'z_transformed'
FILE_SUFFIX = '_whole_brain_correlation_matrix_z_transformed.npz'

Z_THRESHOLD = 4.0
N_FOLDS = 5
RANDOM_STATE = 42

STD_FLOOR = 1e-6

print(f'Matrices root: {WB_ROOT}')
print(f'Splits dir: {SPLITS_DIR}')
print(f'Cohorts CSV: {COHORTS_CSV}')
print(f'GEC checkpoints: {GEC_CHECKPOINT_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

Matrices root: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v3__/matrices
Splits dir: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/metadata/splits_gec
Cohorts CSV: /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__v4__/metadata/cohorts.csv
GEC checkpoints: /mnt/e/fyassine/ad-early-detection/MODEL/notebooks/checkpoints_cost_weighted_gec_whole_brain
Output dir: /mnt/e/fyassine/ad-early-detection/DCI/notebooks/artifacts


## Load Splits

In [3]:
allowed_diagnoses = {'converter', 'mci'}

cohorts_df = pd.read_csv(COHORTS_CSV)
cohorts_df['Repseudonym'] = cohorts_df['Repseudonym'].astype(str)
if 'diagnosis' not in cohorts_df.columns:
    raise ValueError(f"Missing 'diagnosis' column in cohorts CSV: {COHORTS_CSV}")
cohorts_df['diagnosis'] = cohorts_df['diagnosis'].astype(str).str.lower().str.strip()
cohort_diag_map = cohorts_df.groupby('Repseudonym')['diagnosis'].apply(lambda s: set(s)).to_dict()

split_frames = {}
for name in ['train', 'val', 'test']:
    df = pd.read_csv(SPLITS_DIR / f'{name}.csv')
    df['Repseudonym'] = df['Repseudonym'].astype(str)
    if 'diagnosis' not in df.columns:
        raise ValueError(f"Missing 'diagnosis' column in split file: {name}.csv")

    df = df.copy()
    df['diagnosis'] = df['diagnosis'].astype(str).str.lower().str.strip()

    invalid_labels = sorted(set(df['diagnosis']) - allowed_diagnoses)
    if invalid_labels:
        raise ValueError(
            f"Split {name}.csv contains diagnoses outside {sorted(allowed_diagnoses)}: {invalid_labels}"
        )

    missing_in_cohorts = sorted(set(df['Repseudonym']) - set(cohort_diag_map.keys()))
    if missing_in_cohorts:
        raise ValueError(
            f"Split {name}.csv has IDs missing in cohorts.csv (first 10): {missing_in_cohorts[:10]}"
        )

    mismatched = [
        sid for sid, diag in zip(df['Repseudonym'], df['diagnosis'])
        if diag not in cohort_diag_map.get(sid, set())
    ]
    if mismatched:
        raise ValueError(
            f"Split {name}.csv has diagnosis not matched in cohorts.csv (first 10 IDs): {mismatched[:10]}"
        )

    df['converter_status'] = (df['diagnosis'] == 'converter').astype(int)
    split_frames[name] = df

for name, df in split_frames.items():
    counts = df['diagnosis'].value_counts().to_dict()
    n_conv = int(df['converter_status'].sum())
    print(f"{name}: {len(df)} subjects, converters={n_conv}, diagnosis_counts={counts}")

KeyError: 'Repseudonym'

## Discover and Load Correlation Matrices

Replicates the file discovery logic from `ClassificationDataset`:
each subject may have multiple timepoint files (M0, M12, M24, ...).
All matching z-transformed whole-brain files for a subject are loaded.

In [ ]:
all_npz_files = sorted([
    f for f in os.listdir(WB_ROOT)
    if f.endswith(FILE_SUFFIX) and f.startswith('sub-')
])

file_index = {}
for fname in all_npz_files:
    subject_id = fname.split('_')[0].replace('sub-', '')
    file_index.setdefault(subject_id, []).append(fname)

print(f'Total z-transformed files: {len(all_npz_files)}')
print(f'Unique subjects with files: {len(file_index)}')

In [ ]:
def load_matrix(filepath):
    data = np.load(filepath)
    key = 'array' if 'array' in data else list(data.keys())[0]
    return data[key].astype(np.float64)

def load_subject_matrices(subject_id):
    files = file_index.get(subject_id, [])
    matrices = []
    for fname in files:
        mat = load_matrix(WB_ROOT / fname)
        matrices.append(mat)
    return matrices

sample_mat = load_matrix(WB_ROOT / all_npz_files[0])
N_ROIS = sample_mat.shape[0]
print(f'Matrix shape: {sample_mat.shape}')
print(f'Number of ROIs: {N_ROIS}')

In [ ]:
all_subjects_df = pd.concat(split_frames.values(), ignore_index=True)
all_subjects_df = all_subjects_df.drop_duplicates(subset=['Repseudonym'], keep='first')

subject_data = []
missing_subjects = []

for _, row in all_subjects_df.iterrows():
    sid = str(row['Repseudonym'])
    matrices = load_subject_matrices(sid)
    if not matrices:
        missing_subjects.append(sid)
        continue
    for mat in matrices:
        subject_data.append({
            'subject_id': sid,
            'diagnosis': row['diagnosis'],
            'converter_status': int(row['converter_status']),
            'sex': row['sex'],
            'age': float(row['age']),
            'matrix': mat,
        })

print(f'Loaded {len(subject_data)} samples from {len(all_subjects_df)} subjects')
if missing_subjects:
    print(f'Warning: {len(missing_subjects)} subjects had no matching matrix files')

## Build Reference from Training MCI Subjects

Element-wise mean and standard deviation of correlation matrices from training-split
MCI subjects only (non-converters in this strict binary setup). This serves as the
reference distribution for ABI z-score abnormality computation.

In [ ]:
train_ids = set(split_frames['train']['Repseudonym'].astype(str))
reference_train_ids = set(
    split_frames['train'].loc[
        split_frames['train']['diagnosis'].str.lower().str.strip() == 'mci',
        'Repseudonym'
    ].astype(str)
)

if len(reference_train_ids) == 0:
    raise RuntimeError('No MCI subjects found in training split to build ABI reference.')

reference_matrices = []
for sid in reference_train_ids:
    for mat in load_subject_matrices(sid):
        reference_matrices.append(mat)

if len(reference_matrices) == 0:
    raise RuntimeError('No matrices found for training MCI subjects while building ABI reference.')

reference_stack = np.stack(reference_matrices, axis=0)
ref_mean = np.mean(reference_stack, axis=0)
ref_std = np.std(reference_stack, axis=0, ddof=1)

np.fill_diagonal(ref_mean, 0.0)
np.fill_diagonal(ref_std, 1.0)
ref_std = np.clip(ref_std, STD_FLOOR, None)

print(f'Reference built from {len(reference_matrices)} matrices ({len(reference_train_ids)} MCI subjects)')
print(f'ref_mean range: [{ref_mean.min():.4f}, {ref_mean.max():.4f}]')
print(f'ref_std range: [{ref_std.min():.6f}, {ref_std.max():.4f}]')
print(f'Symmetric check (mean): {np.allclose(ref_mean, ref_mean.T)}')
print(f'Symmetric check (std): {np.allclose(ref_std, ref_std.T)}')

## Compute ABI for All Subjects

For each sample's 200×200 correlation matrix:
1. Z-score each connection: `ABS_ij = (F_ij - MEAN_ij) / STD_ij`
2. Threshold: count connections where `|ABS_ij| > 4.0`
3. Abnormality Count (ABC) per ROI: `ABC_i = sum(|ABS_ij| > z, j ≠ i)`
4. ABI = `mean(ABC)` across all ROIs

In [ ]:
def compute_abi(matrix, ref_mean, ref_std, z_threshold=4.0):
    abs_score = (matrix - ref_mean) / ref_std
    abs_score = np.nan_to_num(abs_score, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(abs_score, 0.0)
    abnormal = (np.abs(abs_score) > z_threshold).astype(np.float64)
    abc_per_roi = np.sum(abnormal, axis=1)
    abi = np.mean(abc_per_roi)
    return abi

for entry in subject_data:
    entry['abi'] = compute_abi(entry['matrix'], ref_mean, ref_std, Z_THRESHOLD)

abi_df = pd.DataFrame([{
    'subject_id': e['subject_id'],
    'diagnosis': e['diagnosis'],
    'converter_status': e['converter_status'],
    'sex': e['sex'],
    'age': e['age'],
    'abi': e['abi'],
} for e in subject_data])

print(f'ABI computed for {len(abi_df)} samples')
print(f'\nABI statistics by converter status:')
print(abi_df.groupby('converter_status')['abi'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, group in abi_df.groupby('converter_status'):
    tag = 'Converter' if label == 1 else 'Non-Converter'
    axes[0].hist(group['abi'], bins=30, alpha=0.6, label=tag, edgecolor='black', linewidth=0.4)
axes[0].set_xlabel('ABI')
axes[0].set_ylabel('Count')
axes[0].set_title('ABI Distribution by Converter Status')
axes[0].legend()

sns.violinplot(
    data=abi_df,
    x='converter_status',
    y='abi',
    ax=axes[1],
    inner='quartile',
    palette='Set2',
)
axes[1].set_xticklabels(['Non-Converter', 'Converter'])
axes[1].set_xlabel('Status')
axes[1].set_ylabel('ABI')
axes[1].set_title('ABI Violin Plot')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'abi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Assign Samples to Splits (Matching GEC)

Map each sample to its split using subject IDs. The CV pool = train + val,
holdout = test. Multiple timepoints per subject are included (matching `ClassificationDataset`).

In [ ]:
val_ids = set(split_frames['val']['Repseudonym'].astype(str))
test_ids = set(split_frames['test']['Repseudonym'].astype(str))

def get_split(sid):
    if sid in test_ids:
        return 'test'
    elif sid in val_ids:
        return 'val'
    elif sid in train_ids:
        return 'train'
    return 'unknown'

abi_df['split'] = abi_df['subject_id'].apply(get_split)

cv_df = abi_df[abi_df['split'].isin(['train', 'val'])].reset_index(drop=True)
holdout_df = abi_df[abi_df['split'] == 'test'].reset_index(drop=True)

print(f'CV pool: {len(cv_df)} samples, {cv_df["converter_status"].sum()} converters')
print(f'Holdout: {len(holdout_df)} samples, {holdout_df["converter_status"].sum()} converters')

## 5-Fold Stratified Cross-Validation

Two methods evaluated per fold:
- **Threshold**: Optimal ABI cutoff via Youden's J statistic
- **Logistic Regression**: `LogisticRegression(class_weight='balanced')` on ABI + age + sex

In [ ]:
cv_labels = cv_df['converter_status'].values
cv_abi = cv_df['abi'].values
cv_age = cv_df['age'].values / 100.0
cv_sex = (cv_df['sex'].str.lower() == 'm').astype(float).values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results_threshold = {
    'fold': [], 'val_auc': [], 'val_sensitivity': [],
    'val_specificity': [], 'val_f1': [], 'best_threshold': [],
}
results_logreg = {
    'fold': [], 'val_auc': [], 'val_sensitivity': [],
    'val_specificity': [], 'val_f1': [], 'best_threshold': [],
}

print(f'Starting {N_FOLDS}-fold stratified cross-validation...')
print('=' * 60)

for fold, (train_idx, val_idx) in enumerate(skf.split(cv_abi, cv_labels)):
    print(f'\nFOLD {fold + 1}/{N_FOLDS}')
    print('-' * 40)

    y_train, y_val = cv_labels[train_idx], cv_labels[val_idx]
    abi_train, abi_val = cv_abi[train_idx], cv_abi[val_idx]

    print(f'  Train: {len(train_idx)} samples, {y_train.sum()} converters')
    print(f'  Val:   {len(val_idx)} samples, {y_val.sum()} converters')

    fpr_t, tpr_t, thresholds_t = roc_curve(y_val, abi_val)
    auc_t = auc(fpr_t, tpr_t)
    j_scores = tpr_t - fpr_t
    best_idx = np.argmax(j_scores)
    best_thr = thresholds_t[best_idx]

    preds_t = (abi_val >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds_t, labels=[0, 1]).ravel()
    sens_t = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec_t = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_t = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

    results_threshold['fold'].append(fold + 1)
    results_threshold['val_auc'].append(auc_t)
    results_threshold['val_sensitivity'].append(sens_t)
    results_threshold['val_specificity'].append(spec_t)
    results_threshold['val_f1'].append(f1_t)
    results_threshold['best_threshold'].append(float(best_thr))

    print(f'  [Threshold] AUC={auc_t:.4f}, Sens={sens_t:.4f}, Spec={spec_t:.4f}, F1={f1_t:.4f}, Thr={best_thr:.4f}')

    X_train_lr = np.column_stack([abi_train, cv_age[train_idx], cv_sex[train_idx]])
    X_val_lr = np.column_stack([abi_val, cv_age[val_idx], cv_sex[val_idx]])

    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_train_lr, y_train)
    probs_lr = lr.predict_proba(X_val_lr)[:, 1]

    fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_val, probs_lr)
    auc_lr = auc(fpr_lr, tpr_lr)
    j_scores_lr = tpr_lr - fpr_lr
    best_idx_lr = np.argmax(j_scores_lr)
    best_thr_lr = thresholds_lr[best_idx_lr]

    preds_lr = (probs_lr >= best_thr_lr).astype(int)
    tn_lr, fp_lr, fn_lr, tp_lr = confusion_matrix(y_val, preds_lr, labels=[0, 1]).ravel()
    sens_lr = tp_lr / (tp_lr + fn_lr) if (tp_lr + fn_lr) > 0 else 0.0
    spec_lr = tn_lr / (tn_lr + fp_lr) if (tn_lr + fp_lr) > 0 else 0.0
    f1_lr = 2 * tp_lr / (2 * tp_lr + fp_lr + fn_lr) if (2 * tp_lr + fp_lr + fn_lr) > 0 else 0.0

    results_logreg['fold'].append(fold + 1)
    results_logreg['val_auc'].append(auc_lr)
    results_logreg['val_sensitivity'].append(sens_lr)
    results_logreg['val_specificity'].append(spec_lr)
    results_logreg['val_f1'].append(f1_lr)
    results_logreg['best_threshold'].append(float(best_thr_lr))

    print(f'  [LogReg]    AUC={auc_lr:.4f}, Sens={sens_lr:.4f}, Spec={spec_lr:.4f}, F1={f1_lr:.4f}, Thr={best_thr_lr:.4f}')

print('\n' + '=' * 60)
print('CROSS-VALIDATION COMPLETE')

## Cross-Validation Results Summary

In [ ]:
def summarize_cv(results, method_name):
    print(f'\n{method_name} — CV Summary:')
    print('=' * 60)
    print(f'{"Metric":<20} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
    print('-' * 60)
    for metric in ['val_auc', 'val_sensitivity', 'val_specificity', 'val_f1']:
        vals = np.array(results[metric])
        print(f'{metric:<20} {vals.mean():>10.4f} {vals.std():>10.4f} {vals.min():>10.4f} {vals.max():>10.4f}')
    return {
        'auc_mean': float(np.mean(results['val_auc'])),
        'auc_std': float(np.std(results['val_auc'])),
        'sensitivity_mean': float(np.mean(results['val_sensitivity'])),
        'specificity_mean': float(np.mean(results['val_specificity'])),
        'f1_mean': float(np.mean(results['val_f1'])),
    }

summary_thr = summarize_cv(results_threshold, 'ABI Threshold')
summary_lr = summarize_cv(results_logreg, 'ABI + Logistic Regression')

if summary_lr['auc_mean'] >= summary_thr['auc_mean']:
    best_abi_method = 'logreg'
    best_abi_results = results_logreg
    best_abi_summary = summary_lr
    print('\n=> Selected method: Logistic Regression (higher mean AUC)')
else:
    best_abi_method = 'threshold'
    best_abi_results = results_threshold
    best_abi_summary = summary_thr
    print('\n=> Selected method: Threshold (higher mean AUC)')

## Save ABI Results

Saves all data required by `ABI_VS_GEC.ipynb`:
- `abi_values.csv` — per-sample ABI scores with split assignments
- `abi_baseline_results.json` — CV results for both methods, summaries, and selected method

In [ ]:
abi_df.to_csv(OUTPUT_DIR / 'abi_values.csv', index=False)

run_timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

abi_baseline_results = {
    'run_timestamp': run_timestamp,
    'z_threshold': Z_THRESHOLD,
    'n_folds': N_FOLDS,
    'random_state': RANDOM_STATE,
    'n_rois': int(N_ROIS),
    'reference_group': 'mci_train',
    'n_reference_matrices': len(reference_matrices),
    'n_reference_subjects': len(reference_train_ids),
    'selected_method': best_abi_method,
    'cv_results_threshold': results_threshold,
    'cv_results_logreg': results_logreg,
    'cv_summary_threshold': summary_thr,
    'cv_summary_logreg': summary_lr,
}

with open(OUTPUT_DIR / 'abi_baseline_results.json', 'w') as f:
    json.dump(abi_baseline_results, f, indent=2)

print(f'Saved to: {OUTPUT_DIR}')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}')